# 03C — Audit de projection et calibration spatiale OOF

Ce notebook consomme exclusivement les tables normalisées et sélectionnées de 03B. L'unité d'audit est l'exécution `(model_id, random_state)` ; aucun identifiant de domaine ou d'exécution supplémentaire n'est créé.

## Contrat scientifique

- Vérifier les hashes des huit artefacts 03B consommés contre `checkpoint_manifest.json`.
- Filtrer en mémoire les modèles, runs et seuils effectivement sélectionnés.
- Auditer les projections OOF cibles face aux références train-only, track par track, avec des seuils absolus préspécifiés.
- Calibrer la morphologie uniquement pour les tracks à projection pixel supportés (E3, E4, E7, E8 selon éligibilité).
- Sélectionner exactement un candidat spatial **à l'intérieur de chaque track**, après agrégation égale de ses exécutions sélectionnées. Aucune moyenne, dominance ou pondération ne peut traverser `track_id`.
- Préserver la couche `uncertain` des décisions 3-way pendant tout le post-traitement.
- N'utiliser que les batches 1–2 ; les batches 3–4 sont interdits.
- Produire des artefacts compacts, hashés et vérifiables pour 04A/04C et l'audit.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
if (current_dir / "src").is_dir():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").is_dir():
    PROJECT_ROOT = current_dir.parent
else:
    raise RuntimeError("Launch 03C from the repository or notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.protocol_governance import sha256_file, sha256_payload, verify_frozen_protocol
from src.utils import load_parquet, save_parquet
from src.workflows.projection_domain_audit import (
    build_projection_eligibility,
    build_projection_shift_diagnostics,
)
from src.workflows.simca_calibration_registry import (
    build_selected_execution_registry,
    validate_internal_calibration_manifest,
)
from src.workflows.spatial_postprocessing_calibration import (
    build_spatial_calibration_input,
    build_spatial_candidate_grid,
    calibrate_spatial_postprocessing,
    verify_spatial_postprocessing_lock,
)

In [2]:
RESULTS_TAG = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if expcfg.USE_WAVELENGTH_WINDOW else expcfg.DEFAULT_RESULTS_TAG
)
protocol_dir = PROJECT_ROOT.joinpath(*expcfg.PROTOCOL_ARTIFACT_RELATIVE_DIR)
input_dir = PROJECT_ROOT / "results" / f"{expcfg.INTERNAL_CALIBRATION_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
output_dir = PROJECT_ROOT / "results" / f"{expcfg.DOMAIN_SPATIAL_CALIBRATION_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
output_dir.mkdir(parents=True, exist_ok=True)
input_paths = {
    key: input_dir / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES[key]
    for key in expcfg.DOMAIN_SPATIAL_REQUIRED_03B_ARTIFACTS
}
input_manifest_path = input_dir / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES["checkpoint_manifest"]
output_paths = {
    key: output_dir / filename
    for key, filename in expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES.items()
}

verify_frozen_protocol(protocol_dir, strict=True)
protocol_lock = json.loads(
    (protocol_dir / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]).read_text(encoding="utf-8")
)
protocol_hash = str(protocol_lock["lock_sha256"])
if not input_manifest_path.is_file():
    raise FileNotFoundError(f"Missing 03B manifest: {input_manifest_path}")
input_manifest = json.loads(input_manifest_path.read_text(encoding="utf-8"))
validate_internal_calibration_manifest(
    input_manifest, input_paths,
    required_artifacts=expcfg.DOMAIN_SPATIAL_REQUIRED_03B_ARTIFACTS,
    protocol_hash=protocol_hash,
)

track_contracts = pd.read_parquet(input_paths["track_contracts"])
model_catalog = pd.read_parquet(input_paths["model_catalog"])
selected_models = pd.read_parquet(input_paths["selected_models"])
selected_runs = pd.read_parquet(input_paths["selected_runs"])
selected_threshold_rows = pd.read_parquet(input_paths["selected_thresholds"])
selected_executions, selected_thresholds = build_selected_execution_registry(
    model_catalog, selected_models, selected_runs, selected_threshold_rows,
    track_contracts=track_contracts,
)
expected_track_ids = track_contracts["track_id"].astype(str).tolist()
if set(expected_track_ids) != {f"E{i}" for i in range(1, 9)}:
    raise RuntimeError("The 03B track contract must contain exactly E1-E8.")

oof_objects = pd.read_parquet(
    input_paths["oof_object_predictions"],
    columns=list(expcfg.INTERNAL_CALIBRATION_OOF_OBJECT_COLUMNS),
)
oof_pixels = pd.read_parquet(
    input_paths["oof_pixel_predictions"],
    columns=list(expcfg.INTERNAL_CALIBRATION_OOF_PIXEL_COLUMNS),
)
projection_shift = pd.read_parquet(
    input_paths["projection_shift"],
    columns=list(expcfg.INTERNAL_CALIBRATION_PROJECTION_SHIFT_COLUMNS),
)
object_db, image_db = load_nir_uco_h5(
    PROJECT_ROOT.joinpath(*expcfg.DATABASE_H5_RELATIVE_PATH),
    reconstruct_heavy_object_arrays=False,
)
display(
    selected_executions.groupby(["track_id", "projection_level"], as_index=False).agg(
        n_selected_models=("model_id", "nunique"),
        n_selected_runs=("model_id", "size"),
    )
)

,track_id,projection_level,n_selected_models,n_selected_runs
0,E1,object_projection,2,2
1,E2,object_projection,8,8
2,E3,pixel_projection,1,1
3,E4,pixel_projection,1,1
4,E5,object_projection,1,3
5,E6,object_projection,18,40
6,E7,pixel_projection,1,3
7,E8,pixel_projection,7,21


## Audit train → projection

Les agrégations sont vectorisées par dimension. `overall` et `fold` utilisent uniquement les projections cibles pour l'éligibilité ; les autres strates conservent toute l'information descriptive.

In [3]:
projection_diagnostics = build_projection_shift_diagnostics(
    oof_objects, oof_pixels, selected_executions, projection_shift,
    object_db=object_db, protocol_hash=protocol_hash,
)
projection_eligibility = build_projection_eligibility(
    projection_diagnostics, selected_executions,
    protocol_hash=protocol_hash, expected_track_ids=expected_track_ids,
)
if len(projection_eligibility) != len(expected_track_ids):
    raise RuntimeError("Every track contract must receive one eligibility row.")
if not set(projection_eligibility["eligibility_status"]).issubset(
    set(expcfg.PROJECTION_DOMAIN_ELIGIBILITY_STATUSES)
):
    raise RuntimeError("Unknown projection eligibility status.")
eligibility_diagnostics = projection_diagnostics.loc[
    projection_diagnostics["stratum_type"].isin(
        expcfg.PROJECTION_DOMAIN_ELIGIBILITY_DIMENSIONS
    )
]
if not eligibility_diagnostics["n_observations"].eq(
    eligibility_diagnostics["n_target"]
).all():
    raise RuntimeError("Eligibility diagnostics must use target projections only.")
save_parquet(projection_diagnostics, output_paths["projection_shift_diagnostics"], optimize=False)
save_parquet(projection_eligibility, output_paths["projection_eligibility"], optimize=False)
for track_id in expected_track_ids:
    print(f"\n{track_id} — projection-domain eligibility")
    display(
        projection_eligibility.loc[
            projection_eligibility["track_id"].astype(str).eq(str(track_id))
        ].reset_index(drop=True)
    )



E1 — projection-domain eligibility


,track_id,n_selected_models,n_selected_runs,n_diagnostics,max_abs_standardized_shift,max_out_of_domain_rate,max_target_rejection_rate,eligibility_status,eligibility_reason,rule_version,thresholds_json,protocol_version,protocol_hash
0,E1,2,2,6,0.83237,0.021739,0.021739,eligible,all_predeclared_limits_satisfied,projection_domain_v1,"{""unsupported_max_abs_standardized_shift"":3.0,...",8tracks_v5,af19c5f35d24a81a9c34d0a37c1f2776a31f5aecc70a9b...



E2 — projection-domain eligibility


,track_id,n_selected_models,n_selected_runs,n_diagnostics,max_abs_standardized_shift,max_out_of_domain_rate,max_target_rejection_rate,eligibility_status,eligibility_reason,rule_version,thresholds_json,protocol_version,protocol_hash
0,E2,8,8,24,0.914848,0.065217,0.065217,eligible,all_predeclared_limits_satisfied,projection_domain_v1,"{""unsupported_max_abs_standardized_shift"":3.0,...",8tracks_v5,af19c5f35d24a81a9c34d0a37c1f2776a31f5aecc70a9b...



E3 — projection-domain eligibility


...

,track_id,n_selected_models,n_selected_runs,n_diagnostics,max_abs_standardized_shift,max_out_of_domain_rate,max_target_rejection_rate,eligibility_status,eligibility_reason,rule_version,thresholds_json,protocol_version,protocol_hash
0,E6,18,40,120,1.306352,0.192308,0.192308,eligible_with_warning,out_of_domain_rate;target_rejection_rate,projection_domain_v1,"{""unsupported_max_abs_standardized_shift"":3.0,...",8tracks_v5,af19c5f35d24a81a9c34d0a37c1f2776a31f5aecc70a9b...



E7 — projection-domain eligibility


,track_id,n_selected_models,n_selected_runs,n_diagnostics,max_abs_standardized_shift,max_out_of_domain_rate,max_target_rejection_rate,eligibility_status,eligibility_reason,rule_version,thresholds_json,protocol_version,protocol_hash
0,E7,1,3,9,0.347359,0.024214,0.024214,eligible,all_predeclared_limits_satisfied,projection_domain_v1,"{""unsupported_max_abs_standardized_shift"":3.0,...",8tracks_v5,af19c5f35d24a81a9c34d0a37c1f2776a31f5aecc70a9b...



E8 — projection-domain eligibility


,track_id,n_selected_models,n_selected_runs,n_diagnostics,max_abs_standardized_shift,max_out_of_domain_rate,max_target_rejection_rate,eligibility_status,eligibility_reason,rule_version,thresholds_json,protocol_version,protocol_hash
0,E8,7,21,63,0.432226,0.13805,0.13805,eligible_with_warning,out_of_domain_rate;target_rejection_rate,projection_domain_v1,"{""unsupported_max_abs_standardized_shift"":3.0,...",8tracks_v5,af19c5f35d24a81a9c34d0a37c1f2776a31f5aecc70a9b...


## Calibration spatiale OOF

La clé de carte est `(model_id, random_state, source_image)`. Les seuils 03B sont retirés de l'entrée compacte après calcul des cartes brutes. L'incertitude 3-way reste une couche immuable et est exclue des pixels scorés. Le choix morphologique est ensuite verrouillé séparément pour chaque `track_id` de projection pixel ; les tracks ne partagent aucun critère de sélection.

In [4]:
pixel_track_ids = tuple(map(str, expcfg.SPATIAL_CALIBRATION_PIXEL_TRACK_IDS))
supported_statuses = set(map(str, expcfg.PROJECTION_DOMAIN_SPATIAL_SUPPORTED_STATUSES))

eligible_pixel_tracks = set(
    projection_eligibility.loc[
        projection_eligibility["track_id"].astype(str).isin(pixel_track_ids)
        & projection_eligibility["eligibility_status"].astype(str).isin(supported_statuses),
        "track_id",
    ].astype(str)
)
spatial_track_ids = tuple(
    track_id for track_id in pixel_track_ids if track_id in eligible_pixel_tracks
)
if not spatial_track_ids:
    raise RuntimeError("No eligible pixel-projection track is available for spatial calibration.")

spatial_executions = selected_executions.loc[
    selected_executions["track_id"].astype(str).isin(spatial_track_ids)
    & selected_executions["projection_level"].astype(str).eq("pixel_projection")
].copy()
if set(spatial_executions["track_id"].astype(str)) != set(spatial_track_ids):
    raise RuntimeError(
        "Selected executions do not cover every eligible pixel track: "
        f"expected={sorted(spatial_track_ids)}, "
        f"observed={sorted(set(spatial_executions['track_id'].astype(str)))}."
    )

spatial_input = build_spatial_calibration_input(
    oof_pixels, spatial_executions, selected_thresholds, image_db,
)
observed_batches = set(pd.to_numeric(spatial_input["batch"], errors="raise").astype(int))
if observed_batches.intersection(expcfg.SPATIAL_CALIBRATION_FORBIDDEN_BATCHES):
    raise RuntimeError("Batch 3 or 4 entered spatial calibration.")
if set(spatial_input["truth_level"].astype(str)) != {expcfg.SPATIAL_CALIBRATION_TRUTH_SOURCE}:
    raise RuntimeError("Spatial truth does not match the centralized contract.")

print("Spatial calibration tracks:", list(spatial_track_ids))
for track_id in spatial_track_ids:
    track_input = spatial_input.loc[
        spatial_input["track_id"].astype(str).eq(track_id)
    ]
    display(
        pd.DataFrame(
            [{
                "track_id": track_id,
                "n_selected_models": int(track_input["model_id"].nunique()),
                "n_selected_runs": int(
                    len(track_input[["model_id", "random_state"]].drop_duplicates())
                ),
                "n_images": int(track_input["source_image"].nunique()),
                "n_pixels": int(len(track_input)),
                "uncertain_rate": float(track_input["raw_uncertain"].mean()),
            }]
        )
    )

Spatial calibration tracks: ['E7', 'E8']


,track_id,n_selected_models,n_selected_runs,n_images,n_pixels,uncertain_rate
0,E7,1,3,4,44895,0.0


,track_id,n_selected_models,n_selected_runs,n_images,n_pixels,uncertain_rate
0,E8,7,21,4,314265,0.217371


In [5]:
spatial_grid = build_spatial_candidate_grid()
spatial_metrics, fragment_size_classes, spatial_lock = calibrate_spatial_postprocessing(
    spatial_input, image_db, protocol_hash=protocol_hash, candidate_grid=spatial_grid,
)
save_parquet(spatial_metrics, output_paths["spatial_calibration_metrics"], optimize=False)
save_parquet(fragment_size_classes, output_paths["fragment_size_classes"], optimize=False)
output_paths["spatial_postprocessing_lock"].write_text(
    json.dumps(spatial_lock, indent=2, sort_keys=True), encoding="utf-8",
)
persisted_spatial_metrics = load_parquet(output_paths["spatial_calibration_metrics"])
persisted_fragment_size_classes = load_parquet(output_paths["fragment_size_classes"])
verify_spatial_postprocessing_lock(
    spatial_lock, persisted_spatial_metrics, persisted_fragment_size_classes,
)

print("Spatial selection scope:", spatial_lock["selection_scope"])
print("Spatial selection policy:", spatial_lock["selection_policy"])
selected_parameters_by_track = spatial_lock["selected_parameters_by_track"]
for track_id in spatial_lock["spatial_track_ids"]:
    parameters = dict(selected_parameters_by_track[track_id])
    selected_id = str(parameters["spatial_candidate_id"])
    print(f"\n{track_id} — locked spatial parameters")
    display(pd.DataFrame([parameters]))

    track_metrics = spatial_metrics.loc[
        spatial_metrics["track_id"].astype(str).eq(str(track_id))
    ].copy()
    raw_reference = (
        track_metrics["map_variant"].astype(str).eq("raw")
        & pd.to_numeric(track_metrics["connectivity"], errors="coerce").eq(
            int(parameters["connectivity"])
        )
    )
    locked_candidate = (
        track_metrics["map_variant"].astype(str).eq("postprocessed")
        & track_metrics["spatial_candidate_id"].astype(str).eq(selected_id)
    )
    comparison = track_metrics.loc[raw_reference | locked_candidate]
    display(
        comparison.groupby("map_variant", as_index=False).agg(
            dice=("dice", "mean"),
            iou=("iou", "mean"),
            pixel_recall=("pixel_recall", "mean"),
            component_recall=("component_recall", "mean"),
            component_precision=("component_precision", "mean"),
            smallest_fragment_recall=("smallest_fragment_recall", "mean"),
            split_rate=("split_rate", "mean"),
            merge_rate=("merge_rate", "mean"),
            uncertain_pixel_rate=("uncertain_pixel_rate", "mean"),
        )
    )

Spatial selection scope: within_track
Spatial selection policy: within_track_lexicographic_plateau_then_minimum_complexity

E8 — locked spatial parameters


,spatial_candidate_id,connectivity,morphology_operation,morphology_radius,min_area_pixels
0,spatial_7fc95634a307dc49,1,closing,2,0


,map_variant,dice,iou,pixel_recall,component_recall,component_precision,smallest_fragment_recall,split_rate,merge_rate,uncertain_pixel_rate
0,postprocessed,0.977420,0.955898,0.991670,0.963919,0.485837,0.888313,0.006906,0.0,0.217371
1,raw,0.976674,0.954473,0.989424,0.959359,0.486617,0.872444,0.010211,0.0,0.217371



E7 — locked spatial parameters


,spatial_candidate_id,connectivity,morphology_operation,morphology_radius,min_area_pixels
0,spatial_45d60fb5bd50af48,1,none,0,10


,map_variant,dice,iou,pixel_recall,component_recall,component_precision,smallest_fragment_recall,split_rate,merge_rate,uncertain_pixel_rate
0,postprocessed,0.970419,0.942575,0.984053,1.0,0.867982,1.0,0.000000,0.0,0.0
1,raw,0.911687,0.837903,0.984100,1.0,0.155120,1.0,0.003401,0.0,0.0


## Contrôles de cardinalité et manifeste d'audit

Pour chaque track pixel éligible, son candidat verrouillé doit couvrir exactement toutes ses exécutions sélectionnées. Le manifeste final relie les hashes 03B aux cinq artefacts 03C et enregistre `selected_parameters_by_track`; aucun paramètre spatial global ni pondération inter-track n'est autorisé.

In [6]:
expected_execution_keys = set(
    spatial_executions[["track_id", "model_id", "random_state"]]
    .astype({"track_id": str, "model_id": str})
    .itertuples(index=False, name=None)
)
locked_execution_keys = set(
    spatial_metrics.loc[
        spatial_metrics["is_locked_candidate"].astype(bool),
        ["track_id", "model_id", "random_state"],
    ]
    .astype({"track_id": str, "model_id": str})
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
if locked_execution_keys != expected_execution_keys:
    raise RuntimeError("The track-specific spatial locks do not cover every eligible run.")
if spatial_lock.get("selection_scope") != expcfg.SPATIAL_CALIBRATION_SELECTION_SCOPE:
    raise RuntimeError("The spatial lock is not explicitly within-track.")
if spatial_lock.get("selection_policy") != expcfg.SPATIAL_CALIBRATION_SELECTION_POLICY:
    raise RuntimeError("Unexpected spatial selection policy.")
if "selection_weighting" in spatial_lock or "selected_parameters" in spatial_lock:
    raise RuntimeError("Legacy global spatial-lock fields are forbidden.")
if set(spatial_lock["spatial_track_ids"]) != set(spatial_track_ids):
    raise RuntimeError("Spatial lock and spatial calibration input cover different tracks.")
if set(spatial_lock["selected_parameters_by_track"]) != set(spatial_track_ids):
    raise RuntimeError("Each spatial track must have exactly one locked parameter payload.")
if set(spatial_metrics["map_variant"].astype(str)) != {"raw", "postprocessed"}:
    raise RuntimeError("Raw and postprocessed maps must both be evaluated.")

for track_id in spatial_track_ids:
    expected_id = str(
        spatial_lock["selected_parameters_by_track"][track_id]["spatial_candidate_id"]
    )
    track_locked_ids = set(
        spatial_metrics.loc[
            spatial_metrics["track_id"].astype(str).eq(track_id)
            & spatial_metrics["is_locked_candidate"].astype(bool),
            "spatial_candidate_id",
        ].astype(str)
    )
    if track_locked_ids != {expected_id}:
        raise RuntimeError(
            f"Track {track_id} does not have exactly its own locked candidate."
        )

output_frames = {
    "projection_shift_diagnostics": projection_diagnostics,
    "projection_eligibility": projection_eligibility,
    "spatial_calibration_metrics": persisted_spatial_metrics,
    "fragment_size_classes": persisted_fragment_size_classes,
}
for frame_name, frame in output_frames.items():
    forbidden_identifier_columns = {
        "domain_config_id", "fit_id", "fit_config_id",
        "projection_id", "projection_config_id", "run_id",
    }
    leaked = sorted(forbidden_identifier_columns.intersection(frame.columns))
    if leaked:
        raise RuntimeError(f"{frame_name} leaks redundant identifiers: {leaked}")

artifact_entries = {str(entry["name"]): entry for entry in input_manifest["artifacts"]}
audit_manifest = {
    "protocol_version": expcfg.PROTOCOL_VERSION,
    "schema_version": expcfg.RESULTS_SCHEMA_VERSION,
    "protocol_hash": protocol_hash,
    "projection_rule_version": expcfg.PROJECTION_DOMAIN_AUDIT_RULE_VERSION,
    "spatial_rule_version": expcfg.SPATIAL_CALIBRATION_RULE_VERSION,
    "spatial_selection_scope": spatial_lock["selection_scope"],
    "spatial_selection_policy": spatial_lock["selection_policy"],
    "input_03b_manifest_sha256": sha256_file(input_manifest_path),
    "input_sha256": {
        key: str(artifact_entries[key]["sha256"])
        for key in expcfg.DOMAIN_SPATIAL_REQUIRED_03B_ARTIFACTS
    },
    "selected_counts": {
        "models": int(selected_executions["model_id"].nunique()),
        "runs": int(len(selected_executions)),
        "threshold_rows": int(len(selected_thresholds)),
        "spatial_models": int(spatial_executions["model_id"].nunique()),
        "spatial_runs": int(len(spatial_executions)),
    },
    "spatial_track_ids": list(spatial_lock["spatial_track_ids"]),
    "selected_parameters_by_track": spatial_lock["selected_parameters_by_track"],
    "selected_counts_by_track": spatial_lock["selected_counts_by_track"],
    "natural_execution_key": ["model_id", "random_state"],
    "spatial_decision_key": ["track_id", "spatial_candidate_id"],
    "output_artifacts": {
        key: {
            "path": str(output_paths[key]),
            "row_count": int(len(frame)),
            "columns": list(frame.columns),
            "sha256": sha256_file(output_paths[key]),
        } for key, frame in output_frames.items()
    },
    "spatial_postprocessing_lock_sha256": sha256_file(
        output_paths["spatial_postprocessing_lock"]
    ),
    "spatial_lock_payload_sha256": str(spatial_lock["lock_sha256"]),
}
audit_manifest["manifest_payload_sha256"] = sha256_payload(audit_manifest)
output_paths["audit_manifest"].write_text(
    json.dumps(audit_manifest, indent=2, sort_keys=True), encoding="utf-8",
)
print(
    "03C completed: each eligible pixel track has its own OOF spatial lock; "
    "lineage manifest written for downstream selection."
)

03C completed: each eligible pixel track has its own OOF spatial lock; lineage manifest written for downstream selection.
